In [1]:
# Ensure local project_event_based imports and modules are on sys.path
import sys
from pathlib import Path
repo_root = Path('..').resolve()  # if running from project_event_based/notebooks, adjust accordingly
# Prefer local event-based src then modules
sys.path.insert(0, str((Path.cwd().parent / 'project_event_based' / 'src').resolve()))
sys.path.insert(0, str((Path.cwd().parent / 'modules').resolve()))
print('sys.path head:', sys.path[:3])

sys.path head: ['/Users/xinyan/cours/IFT6759/Splendor/splendor-agent/Splendor-6759/project_event_based/modules', '/Users/xinyan/cours/IFT6759/Splendor/splendor-agent/Splendor-6759/project_event_based/project_event_based/src', '/opt/homebrew/anaconda3/envs/splendor/lib/python314.zip']


In [2]:

import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('seed set to', SEED)

seed set to 42


In [2]:
pip install shimmy

Note: you may need to restart the kernel to use updated packages.


In [1]:
import yaml
import subprocess
from pathlib import Path
import os
import sys

current_script_path = Path(__file__).resolve()
root_path = current_script_path.parent
while root_path.name != 'Splendor-6759' and root_path.parent != root_path:
    root_path = root_path.parent
if root_path.name != 'Splendor-6759':
    root_path = current_script_path.parent.parent
orig_cfg = root_path / 'project_event_based' / 'configs' / 'training' / 'ppo_event_based.yaml'
train_script = root_path / 'project_event_based' / 'scripts' / 'train_maskable.py'

if not orig_cfg.exists():
    print('Original config not found:', orig_cfg)
else:
    tmp_cfg = Path.cwd() / 'tmp_event_config.yaml'
    with open(orig_cfg, 'r') as f:
        cfg = yaml.safe_load(f)


    cfg['training']['total_timesteps'] = 400000
    cfg['training']['tensorboard_log'] = 'logs/tensorboard'
    cfg['ppo']['ent_coef'] = 0.05 
    cfg['ppo']['batch_size'] = 256
    cfg['reward']['event_weights'] = [0.01, 10.0, -0.1, 5.0, 50.0, 0.2, 1.0, 2.0, 5.0]
    cfg['environment']['combine_event_and_score'] = False

    with open(tmp_cfg, 'w') as f:
        yaml.dump(cfg, f)
    

    env = os.environ.copy()
    env["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
    env["PYTHONPATH"] = f"{root_path}/project_event_based/src:{root_path}/modules:" + env.get("PYTHONPATH", "")

    patch_code = (
        "import numpy as np; "
        "np.bool8 = bool; " 
        "import sys, os; "
        "script_path = sys.argv[1]; "
        "sys.argv = sys.argv[1:]; "
        "exec(compile(open(script_path).read(), script_path, 'exec'), {'__file__': script_path, '__name__': '__main__', 'sys': sys, 'os': os, 'np': np})"
    )


    cmd = [sys.executable, '-c', patch_code, str(train_script), '--config', str(tmp_cfg)]
    
    print(' Start...')
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    
    try:
        for line in p.stdout:
            print(line, end='')
    except KeyboardInterrupt:
        p.terminate()
    
    ret = p.wait()
    print('Training process exited with code', ret)

🚀 启动 400k 步实验 (已注入 __file__ 修复)...
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
✅ 底层拦截已激活：已伪造 shimmy 模块以绕过 Discrete(200) 转换报错
Using cpu device
🚀 启动 1M 步实验：控制台将同步显示 episode_reward 和 event_rates 表格...
Logging to ./logs/maskable_v3_1m/MaskablePPO_8
---------------------------------
| events/            |          |
|    event_0_rate    | 0.115    |
|    event_1_rate    | 0.108    |
|    event_2_rate    | 0        |
|    event_3_rate    | 0.0942   |
|    event_4_rate    | 0.00684  |
|    event_5_rate    | 0.115    |
|    event_6_rate    | 0   

In [ ]:
import numpy as np
if not hasattr(np, 'bool8'):
    np.bool8 = bool
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
import subprocess
import sys
from pathlib import Path

model_path = Path('./models/v3_1m_1300000_steps.zip')
eval_script = Path('../scripts/evaluate_model.py')

cmd = [
    sys.executable,  
    str(eval_script), 
    '--model', str(model_path), 
    '--n_games', '1000',
    '--opponent', 'greedy' 
]

print('Evaluate:')
print(' '.join(cmd))
print('--------------------------------------------------')

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in proc.stdout:
        print(line, end='')
except KeyboardInterrupt:
    print("\n Interupt！")
    proc.terminate()
ret = proc.wait()


Evaluate:
/opt/homebrew/anaconda3/envs/splendor/bin/python3.14 ../scripts/evaluate_model.py --model models/v3_1m_1000000_steps.zip --n_games 200 --opponent greedy
--------------------------------------------------
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
✅ Low-level interception activated: Fake shimmy module created to bypass Discrete(200) conversion error
🚀 Start: models/v3_1m_1000000_steps.zip vs greedy
✅ Success: Initialized GreedyAgent with 'value' mode

🏆 Win Rate: 92.50%
📈 Player Score: 15.05
📉 Opponent: 1.42
⚖️ Average Score Dif

In [ ]:
import numpy as np
if not hasattr(np, 'bool8'):
    np.bool8 = bool
import subprocess
import sys
from pathlib import Path

model_path = Path('./models/v3_1m_1300000_steps.zip')
eval_script = Path('../scripts/evaluate_model.py')
cmd = [
    sys.executable, 
    str(eval_script), 
    '--model', str(model_path), 
    '--n_games', '800',
    '--opponent', 'random' 
]

print('Evaluate...')
print('--------------------------------------------------')

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in proc.stdout:
        print(line, end='')
except KeyboardInterrupt:
    proc.terminate()
ret = proc.wait()

Evaluate...
--------------------------------------------------
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
✅ Low-level interception activated: Fake shimmy module created to bypass Discrete(200) conversion error
🚀 Start: models/v3_1m_1000000_steps.zip vs random

🏆 Win Rate: 80.00%
📈 Player Score: 13.43
📉 Opponent: 6.29
⚖️ Average Score Difference: 7.14


In [37]:
!python ../scripts/arena_battle.py

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
✅ Low-level interception activated: Fake shimmy module created to bypass Discrete(200) conversion error
⚔️ Arena officially open: New vs Old model deathmatch...
Game 1 Finished! Winner Info: True
Game 2 Finished! Winner Info: False
Game 3 Finished! Winner Info: True
Game 4 Finished! Winner Info: False
Game 5 Finished! Winner Info: False
Game 6 Finished! Winner Info: False
Game 7 Finished! Winner Info: False
Game 8 Finished! Winner Info: False
Game 9 Finished! Winner Info: True
Game 10 Finished! 